In [1]:
import os
os.chdir("..")
os.makedirs("tests", exist_ok=True)
os.makedirs(".github/workflows", exist_ok=True)
print(os.getcwd())

/home/s2025ae05941/workspace/heart-mlops


In [2]:
%pip install pytest flake8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [flake8]━━━━ 6/7 [flake8]pytest]ig]e]
Note: you may need to restart the kernel to use updated packages.


In [3]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests

Writing pytest.ini


In [4]:
%%writefile .flake8
[flake8]
max-line-length = 100
exclude = .venv,mlruns,notebooks

Writing .flake8


In [5]:

"""Tests for loading the dataset."""
import pandas as pd

from src import train


def test_dataset_has_expected_shape():
    df = pd.read_csv(train.DATA_PATH)
    assert df.shape == (303, 14)


def test_load_data_returns_features_and_binary_target():
    X, y = train.load_data()
    assert list(X.columns) == train.FEATURES
    assert set(y.unique()) == {0, 1}
    assert len(X) == len(y) == 303


def test_missing_values_only_in_ca_and_thal():
    X, _ = train.load_data()
    columns_with_missing = set(X.columns[X.isna().any()])
    assert columns_with_missing <= {"ca", "thal"}

/home/s2025ae05941/workspace/heart-mlops/.venv/lib64/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/s2025ae05941/workspace/heart-mlops/.venv/lib64/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


In [6]:

"""Tests for the preprocessing pipeline and model training."""
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src import train


def test_preprocessor_fills_missing_values_and_encodes():
    X, _ = train.load_data()
    Z = train.build_preprocessor().fit_transform(X)
    if hasattr(Z, "toarray"):  # sparse matrix -> dense
        Z = Z.toarray()
    assert Z.shape[0] == len(X)
    assert Z.shape[1] > X.shape[1]  # one-hot encoding adds columns
    assert not np.isnan(Z).any()  # no missing values left


def test_pipeline_trains_and_scores_well():
    X, y = train.load_data()
    pipe = Pipeline([("prep", train.build_preprocessor()),
                     ("model", LogisticRegression(max_iter=5000))])
    pipe.fit(X, y)
    scores = train.test_scores(pipe, X, y)
    assert set(scores) == set(train.METRICS)
    assert scores["roc_auc"] > 0.8


def test_candidate_models_include_two_models():
    models = train.candidate_models()
    assert {"logistic_regression", "random_forest"} <= set(models)

In [8]:
%%writefile .github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:

jobs:
  lint-and-test:
    name: Lint and unit tests
    runs-on: ubuntu-latest
    steps:
      - name: Check out code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.9"
          cache: pip

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install flake8 pytest

      - name: Lint with flake8
        run: flake8 src tests

      - name: Run unit tests
        run: pytest -v --junitxml=test-results.xml

      - name: Upload test results
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: test-results
          path: test-results.xml

  train:
    name: Train model
    needs: lint-and-test
    runs-on: ubuntu-latest
    steps:
      - name: Check out code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.9"
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Train and evaluate models
        run: python src/train.py | tee training-log.txt

      - name: Upload model, plots and logs
        uses: actions/upload-artifact@v4
        with:
          name: trained-model
          path: |
            models/
            reports/figures/
            training-log.txt
            mlruns/

Writing .github/workflows/ci.yml


In [9]:
!flake8 src tests && echo "LINT OK"

src/download_data.py:29:11: W292 no newline at end of file
src/eda.py:73:11: W292 no newline at end of file


In [10]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.9.21, pytest-8.4.2, pluggy-1.6.0 -- /home/s2025ae05941/workspace/heart-mlops/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/s2025ae05941/workspace/heart-mlops
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.12.1
collected 0 items                                                              

============================ no tests ran in 0.19s =============================
